In [ ]:
# import necessary packages
import folium
from pathlib import Path
import pandas as pd
import geopandas as gpd
import platform


In [ ]:
# read in the wetlands datasets from previous processing work in wetland_explore

# establish paths to datasets
project_fldr = Path.cwd().parents[1]
raw_fldr = project_fldr / 'data' / 'raw'
processed_fldr = project_fldr / 'data' / 'processed'
figure_fldr = project_fldr/"figures"

# set a path to the main box folder based on machine's os
# if windows: 
if platform.system() == 'Windows':
    box_path = Path.home() / 'Box'
# if mac:
else:
    box_path = Path.home() / "Library" / "CloudStorage" / "Box-Box"

#Confirm that the box path is valid
if box_path.exists():
    print(f'Box folder found at {box_path}')
else:
    print(f'Failed to locate the Box folder.\nIs the BoxDrive app running?\nAre you logged in?')

# create path object that goes to the specific folder on Box drive
# uploading wetlands that are bigger than a hectare
html_map_path = box_path / 'ENVIRON 712L-Students' / 'Data' / 'BunkerAndersonTseng' / 'html_maps'


# palustrine and riverine to come later
ca_outline = gpd.read_file(processed_fldr/'wetland_explore'/"ca_outline.geojson")
marine = gpd.read_file(processed_fldr/'wetland_explore'/"marine_simpler.geojson")
estuarine = gpd.read_file(processed_fldr/'wetland_explore'/"estuarine_simpler.geojson")
lacustrine = gpd.read_file(processed_fldr/'wetland_explore'/"lacustrine_simpler.geojson")



Only using Marine Estuarine and Lacustrine right now due to smaller data sizes, will add in R+P later when dashboard is more fleshed out.

In [ ]:
# DEFINE MAP PROPERTIES

# bounding coordinates for California and rest of map
minx, miny, maxx, maxy = ca_outline.total_bounds

#plot wetlands, by system color
system_colors = {
    "Marine": "#02518a",
    "Estuarine": "#0b2ce9",
    "Riverine": "#17becf",
    "Lacustrine": "#01f694",
    "Palustrine": "#4bcb00"
}

In [ ]:
# MAKE THE EXPLORE

# create the actual map as a base now
m = folium.Map(
    zoom_start=6,
    tiles=None
)

# ESRI satellite basemap
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, Maxar, Earthstar Geographics, and the GIS User Community",
    name="Satellite Imagery",
    overlay=False,
    control=False
).add_to(m)

# this should leave us so that the initial zoom is the outline of california
m.fit_bounds([
    [miny, minx],  # Southwest corner
    [maxy, maxx]   # Northeast corner
])

# add CA outline
folium.GeoJson(
    ca_outline.to_json(),
    style_function=lambda x: {
        "color": "black",
        "fillOpacity": 0,
        "weight": 2
    },
    control= False # always on
).add_to(m)

# Marine layer
folium.GeoJson(
    marine.to_json(),
    name="Marine System Wetlands",
    style_function=lambda x: {
        "color": system_colors["Marine"],
        "fillColor": system_colors["Marine"],
        "fillOpacity": 0.9,
        "weight": 0.5
    },
    control=True #can be toggled
).add_to(m)

# Estuarine layer
folium.GeoJson(
    estuarine.to_json(),
    name="Estuarine System Wetlands",
    style_function=lambda x: {
        "color": system_colors["Estuarine"],
        "fillColor": system_colors["Estuarine"],
        "fillOpacity": 0.9,
        "weight": 0.5
    },
    control=True #can be toggled
).add_to(m)

# Lacustrine layer
folium.GeoJson(
    lacustrine.to_json(),
    name="Lacustrine System Wetlands",
    style_function=lambda x: {
        "color": system_colors["Lacustrine"],
        "fillColor": system_colors["Lacustrine"],
        "fillOpacity": 0.9,
        "weight": 0.5
    },
    control=True #can be toggled
).add_to(m)

# Add layer control
folium.LayerControl(collapsed=False).add_to(m)

m

In [ ]:
# save to box
# m.save(html_map_path/"wetland_map.html")